### Packages for the Data Generation and Mopdelling of PD (Probability of Default), LGD (Loss Given Default) and EAD (Exposure at Default)

In [176]:
# Data Management and Processing
import pandas as pd
import numpy as np
import scipy
import random

In [177]:
# Machine Learning and Statistics
import sklearn
import statsmodels.api as sm
import tensorflow as tf

### Data generator

##### Support functions

In [178]:
############# Function to create a profession based on the educational level #########################
def generate_profession(education):
    if education == "high school or lower":
        return random.choices(["LowSkilled", "Unemployed_LowSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education == "ausbildung":
        return random.choices(["MediumSkilled", "Unemployed_MediumSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education in ["bachelor degree", "post graduate degree"]:
        return random.choices(["HighSkilled", "Unemployed_HighSkilled"], weights=[0.9, 0.1], k=1)[0]

In [179]:
############################### Function to generate monthly income and expenditure ##################################                

# Define income parameters for different profession levels and age ranges
income_parameters = {
    ("LowSkilled", "Unemployed_LowSkilled"): {
        (30, 35): {"mean": 1000, "std_dev": 200, "max_income": 2000},
        (36, 40): {"mean": 1200, "std_dev": 200, "max_income": 2400},
        (41, 45): {"mean": 1500, "std_dev": 250, "max_income": 3000},
        (46, 50): {"mean": 1800, "std_dev": 300, "max_income": 3600},
        (51, 55): {"mean": 2000, "std_dev": 350, "max_income": 4000},
        (56, 60): {"mean": 2200, "std_dev": 400, "max_income": 4300},
        (61, 65): {"mean": 2400, "std_dev": 600, "max_income": 4500},
    },
    ("MediumSkilled", "Unemployed_MediumSkilled"): {
        (30, 35): {"mean": 1800, "std_dev": 300, "max_income": 4000},
        (36, 40): {"mean": 2300, "std_dev": 400, "max_income": 5000},
        (41, 45): {"mean": 2600, "std_dev": 500, "max_income": 6000},
        (46, 50): {"mean": 3000, "std_dev": 600, "max_income": 7000},
        (51, 55): {"mean": 3500, "std_dev": 800, "max_income": 8000},
        (56, 60): {"mean": 4000, "std_dev": 800, "max_income": 9000},
        (61, 65): {"mean": 5000, "std_dev": 1000, "max_income": 10000},
    },
    ("HighSkilled", "Unemployed_HighSkilled"): {
        (30, 35): {"mean": 3000, "std_dev": 500, "max_income": 10000},
        (36, 40): {"mean": 4500, "std_dev": 700, "max_income": 15000},
        (41, 45): {"mean": 6000, "std_dev": 1000, "max_income": 20000},
        (46, 50): {"mean": 7000, "std_dev": 1500, "max_income": 25000},
        (51, 55): {"mean": 8000, "std_dev": 2000, "max_income": 30000},
        (56, 60): {"mean": 9000, "std_dev": 3000, "max_income": 40000},
        (61, 65): {"mean": 10000, "std_dev": 4000, "max_income": 50000},
    }
}

# To calculate the monthly income and expenditure
def generate_income_expense(profession_undertake, current_age, num_dependents):
    # Iterate over income parameters for each profession group
    for profession_group, age_ranges in income_parameters.items():
        # Check if profession_undertake is one of the professions in the profession_group tuple
        if profession_undertake in profession_group:
            # Iterate over the age ranges and income parameters
            for age_range, params in age_ranges.items():
                if age_range[0] <= current_age <= age_range[1]:
                    mean_income = params["mean"]
                    std_dev = params["std_dev"]
                    max_income = params["max_income"]
                    unemployement_money = mean_income * 0.5  # 50% of mean income for unemployment

                    # If profession_undertake contains the word "Unemployed" before "_", return the unemployment money
                    if profession_undertake.split("_")[0] == "Unemployed":
                        income = unemployement_money
                        expenditure = generate_expenditure(income, mean_income, num_dependents) # The belong to the population under mean_income
                        return income, expenditure
                    
                    # If profession_undertake does not contain the word "Unemployed", generate income with an specific rule
                    else:
                        calc_income = int(np.random.normal(mean_income, std_dev))
                        income = max(unemployement_money, min(calc_income, max_income))  
                        expenditure = generate_expenditure(income, mean_income, num_dependents)
                        return income, expenditure

# Support function to calculate the expenditure
def generate_expenditure(income, mean_income, num_dependents):
    # Values for low and for high income people (lower possible value, mode, higher possible value) depending on the number of dependents
    low_income_params = [(0.5, 0.7, 1.5), (0.7, 0.8, 1.5), (0.8, 0.9, 1.5), (0.9, 0.9, 1.5), (0.9, 1.0, 1.5)]
    high_income_params = [(0.5, 0.6, 1.5), (0.6, 0.65, 1.5), (0.7, 0.75, 1.5), (0.7, 0.75, 1.5), (0.8, 0.85, 1.5)]
    # The parameters that should be taken depend on whether the income is below or above the mean income
    params = low_income_params if income < mean_income else high_income_params
    # Here we recover the parameters
    left, mode, right = params[min(num_dependents, 4)]  # Ensure index stays within range
    # The expenditure is calculated given a rule of min, mode, max
    expenditure = income * np.random.triangular(left=left, mode=mode, right=right)
    return expenditure



In [180]:
############################### Function to generate the credit to be requested ##################################
# Note the credits will be exactly for one year
def generate_credit_requested(income): ### It will depend on the income
    # The will maximum enter as for a credit that represent between 25% and 150% of their income and the probabilities of any values are equal, so a uniform distribution
    percentage_of_monthly_income = random.uniform(0.25, 1.5)
    yearly_income = income * 12 # must be changes once dynamically made
    credit_requested_yearly = yearly_income *percentage_of_monthly_income
    credit_requested_monthly = credit_requested_yearly / 12
    percentage_credit_month_income = credit_requested_monthly/income
    return credit_requested_monthly, percentage_credit_month_income
    

In [181]:
####################### A function to calculate the collateral ########################
def calculate_collateral(profession):
    if "HighSkilled" in profession: # If the person is HighSkilled, even if currently unemployed
        # random.random() returns a float number between 0 and 1
        if random.random() < 0.10: # 10 % of the people do not have collateral
            return 0
        else: # 90 % have a collateral between 10,000 and 80,000
            return random.randint(10000, 80000)
    elif "MediumSkilled" in profession: # If the person is HighSkilled, even if currently unemployed
        if random.random() < 0.20: # 20 % of the people do not have collateral
            return 0
        else: # 80 % have a collateral between 10,000 and 50,000
            return random.randint(10000, 50000)
    else: # If Lowskilled
        if random.random() < 0.35: # 35 % of the people do not have collateral
            return 0
        else: # 65 % have a collateral between 5,000 and 25,000
            return random.randint(5000, 25000)

In [182]:
####################### A function to estimate the seizable assets to calculate the LGD ########################
def estimate_seizable_assets(monthly_income, savings, profession, collateral):
    
    # Base asset estimation as a portion of income and savings
    # This is a proxy, people with higher income, tend to have higher assets
    # If the savinds are negative, and higher than 2 times the monthy income this will draw this to be negative
    # In the end, if no assets can be seizured, it means that maybe it is not convenient for the bank to actually give the loan
    base_asset = savings + 2 * monthly_income

    # People that are High-Skilled tend to have more assets to be seized, even if they are currently unemployed
    if "HighSkilled" in profession:
        base_asset *= 1.2
    elif "MediumSkilled" in profession:
        base_asset *= 1.0
    else:
        base_asset *= 0.8

    # If they have a collateral, more sizes are
    base_asset += collateral

    # Add some noise to simulate unpredictability
    # That means given some processes in normal life, it is not always sure that 100% of the collateral can be recovered without cost
    base_asset *= np.random.normal(1, 0.1)  # 10% variation

    # To ensure that if nothing can be seized, because in the end a negative value is returned, then we get a zero
    return max(base_asset, 0)

In [183]:
############################### Function to generate the whether the person defaults or not ##################################
def generate_default_label(profession, past_credits, debt_to_income_ratio_before_credit, credit_to_income_ratio):
    """This simulates a default label (0/1) based on financial risk factors."""
    """What we will use will be the """
    
    # Configurable risk settings per profession
    risk_settings = {
        "Unemployed_LowSkilled":     {"base": 0.15, "weights": (0.30, 0.6, 0.4)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "LowSkilled":                {"base": 0.08, "weights": (0.20, 0.5, 0.3)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "Unemployed_MediumSkilled": {"base": 0.12, "weights": (0.30, 0.5, 0.3)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "MediumSkilled":            {"base": 0.05, "weights": (0.20, 0.45, 0.25)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "Unemployed_HighSkilled":   {"base": 0.09, "weights": (0.30, 0.45, 0.25)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "HighSkilled":              {"base": 0.2,  "weights": (0.20, 0.4, 0.2)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
    }

    # setting variable is sett to retrieve the element of reisk_settings for the profession given
    settings = risk_settings.get(profession)
    # In case there is the profession given for the function does not match any of the professions above listed
    if not settings:
        raise ValueError(f"Unknown profession: {profession}")
    
    # Defining the weigths for the calcualtion of the probability of default
    w1, w2, w3 = settings["weights"]
    # Defining the base probability of default
    base = settings["base"]

    # Calculating the risk factor with the weigths
    risk_factor = past_credits * w1 + debt_to_income_ratio_before_credit * w2 + credit_to_income_ratio * w3
    # Defining the default probability
    default_probability = min(1, base + risk_factor)

    # We want a non-deterministic y-categorical variable that will make that same profiles will not always lead to the same result
    # So even if two people may fall on the same profile, maybe they will not default
    # return 1 if random.random() < default_probability else 0
    return int(random.random() < default_probability)

##### Data Generator for the original state of individuals

In [184]:
# Function to generate data accordingly to some requirements
def data_generator(number_of_customers):
    data = []
    for i in range(number_of_customers):
        
        # ---------------- X-Variables -------------------------------#
        ##### Variables not directly dependent on other variables #####
        name = f"name{i}" # names are created according to the index "i"
        age = random.randint(30, 60) # As the maximum attainable age that we want in the game is 65
        education_level = random.choices(["high school or lower", "ausbildung", "bachelor degree", "post graduate degree"],  weights=[0.3, 0.3, 0.3, 0.1], k=1)[0]
        # Number of unpaid past credits
        past_credits = random.choices([0, 1, 2, 3], weights=[0.6, 0.3, 0.08, 0.02], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        # Number of dependents
        dependents = random.choices([0, 1, 2, 3, 4], weights=[0.6, 0.3, 0.06, 0.03, 0.01], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        
        ##### Variables directly dependent on other variables #####
        # Generate profession based on education level
        profession = generate_profession(education_level)
        # Generate monthly income based on profession, age and number of dependents
        monthly_income = generate_income_expense(profession, age, dependents)[0]
        # Generate monthly expenditure dependening on the income, mean income and number of dependents
        monthly_expenditure = generate_income_expense(profession, age, dependents)[1]
        
        ##### Other variables generated from the variables above #####
        savings_debt = monthly_income - monthly_expenditure
        
        # Debt to income ratio: PARTIAL, before the credit
        if savings_debt < 0:
            #debt_to_income_ratio = f"{abs(savings_debt/monthly_income):.2%}"
            debt_to_income_ratio_partial = abs(savings_debt/monthly_income)
        else:
            #debt_to_income_ratio = f"{0:.2%}"
            debt_to_income_ratio_partial = 0
        
        # ---------------- Credit amount requested and time of the request--------------------------------#
        monthly_credit = generate_credit_requested(monthly_income)[0]
        credit_to_income_ratio = generate_credit_requested(monthly_income)[1]
        
        # calculating the requested duration of the loan (maybe we can make it to be then also set by the bank whether it accepts it up to this term or not)
        if 0.25 <= credit_to_income_ratio < 0.5: 
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.2, 0.3, 0.2, 0.2, 0.01], k=1)[0]
        elif 0.5 <= credit_to_income_ratio < 0.75: 
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.1, 0.2, 0.2, 0.2, 0.3], k=1)[0]
        elif 0.75 <= credit_to_income_ratio < 1.0: 
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.08, 0.12, 0.2, 0.25, 0.35], k=1)[0]
        else:
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.02, 0.08, 0.2, 0.2, 0.5], k=1)[0]
        
        # calculating the monthly debt if the monthluy credit is issued
        debt_after_credit = savings_debt - monthly_credit
        if debt_after_credit > 0: # if still the monthly savings are higher than the credit, then the total debt to incom ratio should be zero
            debt_to_income_ratio_total = 0
        else: 
            debt_to_income_ratio_total = abs(debt_after_credit/monthly_income)
            
        # collateral and seizurable asset
        collateral = calculate_collateral(profession)
        estimated_seizable_assets = estimate_seizable_assets(monthly_income, savings_debt, profession, collateral)
        
        # ---------------- Y-Variable --------------------------------#
        default_not_default = generate_default_label(profession, past_credits, debt_to_income_ratio_partial, credit_to_income_ratio)
        
        data.append({
            'name': name, # independent
            'age': age, # independent
            'educational level': education_level, # independent
            'number of not paid past credits': past_credits, # independent
            'dependents': dependents, # independent
            'profession': profession, # depends on education
            'monthly income': monthly_income, # depends on profession and age
            'monthly expenditure': monthly_expenditure, # depends on income, mean income per age and profession, and the number of dependents
            'savings (debt)': savings_debt, # monthly income - monthly expenditure
            'debt-to-income ratio before credit': debt_to_income_ratio_partial, # abs(savings_debt/monthly_income)
            'credit: monthly amount': monthly_credit, # depends on the income
            'credit-to-income ratio': credit_to_income_ratio, # credit/income
            'requested_loan_duration': credit_term_months, # depends on the credit-to-income ratio
            'debt-to-income ratio after credit': debt_to_income_ratio_total, # abs((savings_debt - credit)/monthly_income)
            'collateral': collateral, # depends on the profession
            'estimated seizable assets': estimated_seizable_assets, # it is based on the monthly income, the savings, the profession and the collateral
            'y-categorical-default': default_not_default # depending on profession, past_credits, debt_to_income_ratio_partial, credit_to_income_ratio
        })

    # Create a pandas DataFrame
    df = pd.DataFrame(data)
    return df

##### Generating one data frame

In [185]:
number_of_customers_1 = 1000
df_1 = data_generator(number_of_customers_1)
df_1

,name,age,educational level,number of not paid past credits,dependents,profession,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,debt-to-income ratio after credit,collateral,estimated seizable assets,y-categorical-default
0,name0,42,high school or lower,1,0,LowSkilled,1224.0,1506.545590,-282.545590,0.230838,515.719196,1.345488,60,0.652177,7953,10099.582841,1
1,name1,44,high school or lower,3,1,LowSkilled,1896.0,1591.271515,304.728485,0.000000,1272.573333,1.008021,24,0.510467,15634,22089.233754,1
2,name2,30,high school or lower,0,0,LowSkilled,1033.0,803.987260,229.012740,0.000000,1153.501625,0.992579,48,0.894955,20844,18041.798287,0
3,name3,41,post graduate degree,0,1,HighSkilled,6992.0,4667.171446,2324.828554,0.000000,4666.184651,1.193705,60,0.334862,62777,80893.005144,0
4,name4,30,high school or lower,0,0,LowSkilled,1070.0,868.404229,201.595771,0.000000,1410.257475,1.292106,60,1.129590,6951,8949.128105,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,56,bachelor degree,0,0,HighSkilled,4500.0,5459.236327,-959.236327,0.213164,1565.422592,0.989129,36,0.561035,12912,24260.034784,1
996,name996,45,ausbildung,0,0,MediumSkilled,2436.0,1926.002775,509.997225,0.000000,2650.327579,0.684856,48,0.878625,0,5171.754976,0
997,name997,49,post graduate degree,0,0,Unemployed_HighSkilled,3500.0,2637.012330,862.987670,0.000000,3692.501195,0.717593,12,0.808432,56493,69052.133078,1
998,name998,41,ausbildung,0,1,MediumSkilled,2550.0,2213.388587,336.611413,0.000000,2142.087015,1.371900,48,0.708030,0,5226.054445,0


##### DF statistics

In [186]:
df_1.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,debt-to-income ratio after credit,collateral,estimated seizable assets,y-categorical-default
count,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,44.78400,0.532000,0.535000,3612.168000,3329.364581,282.803419,0.106481,3164.658875,0.887691,43.308000,0.822880,27614.306000,35813.991376,0.498000
std,8.94804,0.737238,0.812056,2560.136893,2658.096965,1697.797528,0.227843,2698.608145,0.357755,15.973487,0.481638,22753.506356,27038.423536,0.500246
min,30.00000,0.000000,0.000000,500.000000,300.910488,-10698.896596,0.000000,184.585364,0.250541,12.000000,0.000000,0.000000,561.971358,0.000000
25%,37.00000,0.000000,0.000000,1749.750000,1475.213862,-246.916409,0.000000,1252.930880,0.570547,36.000000,0.466365,10715.500000,14873.622924,0.000000
50%,44.00000,0.000000,0.000000,2836.000000,2562.642602,241.563669,0.000000,2298.475855,0.898287,48.000000,0.798866,22630.500000,29007.856732,0.000000
75%,52.25000,1.000000,1.000000,4727.500000,4188.166526,851.489628,0.118208,4305.044396,1.187782,60.000000,1.149186,42660.500000,52468.411323,1.000000
max,60.00000,3.000000,4.000000,16008.000000,17432.801983,10725.032818,1.757010,18171.700020,1.499827,60.000000,2.905556,79960.000000,121104.313590,1.000000


Monthly income by profession

In [187]:
df_1.groupby('profession')['monthly income'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,377.0,5998.636605,2532.938923,2048.0,4144.0,5765.0,7136.0,16008.0
LowSkilled,260.0,1550.980769,494.584639,574.0,1167.0,1505.0,1929.0,2934.0
MediumSkilled,267.0,2877.254682,968.503894,939.0,2139.5,2806.0,3462.0,5808.0
Unemployed_HighSkilled,37.0,3027.027027,1052.087514,1500.0,2250.0,3000.0,4000.0,4500.0
Unemployed_LowSkilled,30.0,788.333333,179.886618,500.0,600.0,825.0,900.0,1100.0
Unemployed_MediumSkilled,29.0,1501.724138,413.712020,900.0,1150.0,1500.0,2000.0,2000.0


Debt-to-income ratio by profession

In [188]:
df_1.groupby('profession')['debt-to-income ratio before credit'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,377.0,0.133614,0.270409,0.0,0.0,0.0,0.162759,1.649029
LowSkilled,260.0,0.093688,0.204904,0.0,0.0,0.0,0.108894,1.757010
MediumSkilled,267.0,0.100164,0.212765,0.0,0.0,0.0,0.094933,1.639782
Unemployed_HighSkilled,37.0,0.038892,0.081536,0.0,0.0,0.0,0.042199,0.386211
Unemployed_LowSkilled,30.0,0.067670,0.117509,0.0,0.0,0.0,0.079911,0.458360
Unemployed_MediumSkilled,29.0,0.052992,0.081004,0.0,0.0,0.0,0.091394,0.237582


In [189]:
df_1.groupby('profession')['debt-to-income ratio after credit'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,377.0,0.854874,0.507417,0.000000,0.507734,0.810940,1.175859,2.530267
LowSkilled,260.0,0.816058,0.467957,0.000000,0.465277,0.815299,1.137921,2.905556
MediumSkilled,267.0,0.773927,0.488600,0.000000,0.388671,0.730048,1.121239,2.393373
Unemployed_HighSkilled,37.0,0.927756,0.381062,0.085234,0.731235,0.918601,1.230589,1.580638
Unemployed_LowSkilled,30.0,0.771590,0.344853,0.000000,0.513942,0.759514,0.972590,1.453959
Unemployed_MediumSkilled,29.0,0.838080,0.401259,0.024347,0.441931,0.945118,1.163433,1.459197


In [190]:
df_1.groupby('profession')['credit-to-income ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,377.0,0.912351,0.346843,0.257536,0.634736,0.921722,1.204880,1.499827
LowSkilled,260.0,0.896074,0.367974,0.257964,0.564190,0.939349,1.204469,1.498842
MediumSkilled,267.0,0.847928,0.362289,0.250965,0.527897,0.835612,1.150799,1.489515
Unemployed_HighSkilled,37.0,0.801745,0.363052,0.256166,0.530673,0.786379,1.026947,1.455978
Unemployed_LowSkilled,30.0,1.019286,0.280487,0.448584,0.830442,1.072439,1.233311,1.476381
Unemployed_MediumSkilled,29.0,0.831556,0.380987,0.250541,0.502654,0.834664,1.135263,1.488720


In [191]:
df_1.groupby('profession')['collateral'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,377.0,43829.517241,23292.517363,0.0,24731.0,44840.0,64994.00,79960.0
LowSkilled,260.0,9574.542308,8337.120642,0.0,0.0,9385.0,16517.00,24930.0
MediumSkilled,267.0,22804.808989,15428.010836,0.0,11852.0,23284.0,35277.50,49764.0
Unemployed_HighSkilled,37.0,39268.918919,23158.958129,0.0,22083.0,34556.0,60832.00,79068.0
Unemployed_LowSkilled,30.0,11294.700000,9038.724087,0.0,0.0,12987.5,19494.25,23733.0
Unemployed_MediumSkilled,29.0,24845.586207,15881.632260,0.0,16741.0,26632.0,39298.00,47226.0


In [192]:
df_1.groupby('profession')['estimated seizable assets'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,377.0,58695.083588,25182.944009,5996.978392,40177.769991,59525.612733,76938.084530,121104.313590
LowSkilled,260.0,12068.783209,8309.504349,561.971358,3148.824656,12273.281128,19089.454666,29802.279670
MediumSkilled,267.0,28725.016955,15856.301031,2072.177672,17487.373234,27394.360392,41678.852716,70511.061427
Unemployed_HighSkilled,37.0,46214.583383,22315.461717,6052.965305,26920.298553,46636.961585,65994.283696,79909.064639
Unemployed_LowSkilled,30.0,12336.701969,9035.261685,799.433843,1490.157814,14038.507895,19796.157574,26678.289712
Unemployed_MediumSkilled,29.0,27532.450285,15719.741816,2106.569368,17300.729111,31201.064570,40465.272766,52461.190672


In [193]:
df_1.groupby('profession')['y-categorical-default'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,377.0,0.511936,0.500522,0.0,0.00,1.0,1.0,1.0
LowSkilled,260.0,0.580769,0.494385,0.0,0.00,1.0,1.0,1.0
MediumSkilled,267.0,0.397004,0.490196,0.0,0.00,0.0,1.0,1.0
Unemployed_HighSkilled,37.0,0.405405,0.497743,0.0,0.00,0.0,1.0,1.0
Unemployed_LowSkilled,30.0,0.733333,0.449776,0.0,0.25,1.0,1.0,1.0
Unemployed_MediumSkilled,29.0,0.379310,0.493804,0.0,0.00,0.0,1.0,1.0


# MODELLING PD, LGD, EAD

## PD (probability of default)
The probability that a customer with default at some point

#### Packages

In [194]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#### Converting variables into dummies

In [195]:
# Convert categorical variables to numeric
df_R = pd.get_dummies(df_1, columns=["educational level", "profession"], drop_first=True)
df_R

,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,estimated seizable assets,y-categorical-default,educational level_bachelor degree,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled
0,name0,42,1,0,1224.0,1506.545590,-282.545590,0.230838,515.719196,1.345488,...,10099.582841,1,False,True,False,True,False,False,False,False
1,name1,44,3,1,1896.0,1591.271515,304.728485,0.000000,1272.573333,1.008021,...,22089.233754,1,False,True,False,True,False,False,False,False
2,name2,30,0,0,1033.0,803.987260,229.012740,0.000000,1153.501625,0.992579,...,18041.798287,0,False,True,False,True,False,False,False,False
3,name3,41,0,1,6992.0,4667.171446,2324.828554,0.000000,4666.184651,1.193705,...,80893.005144,0,False,False,True,False,False,False,False,False
4,name4,30,0,0,1070.0,868.404229,201.595771,0.000000,1410.257475,1.292106,...,8949.128105,0,False,True,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,56,0,0,4500.0,5459.236327,-959.236327,0.213164,1565.422592,0.989129,...,24260.034784,1,True,False,False,False,False,False,False,False
996,name996,45,0,0,2436.0,1926.002775,509.997225,0.000000,2650.327579,0.684856,...,5171.754976,0,False,False,False,False,True,False,False,False
997,name997,49,0,0,3500.0,2637.012330,862.987670,0.000000,3692.501195,0.717593,...,69052.133078,1,False,False,True,False,False,True,False,False
998,name998,41,0,1,2550.0,2213.388587,336.611413,0.000000,2142.087015,1.371900,...,5226.054445,0,False,False,False,False,True,False,False,False


#### Defining X and y

In [196]:
# Column "name" is dropped from the dataframe, no need to keep it
# All the variables except y-categorical-default are X
X = df_R.drop(columns=["name", "y-categorical-default"])
y = df_R["y-categorical-default"]

#### Split between trainning and test sets

In [197]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#### Standardize the variables for a better gradient descendt
Standardizing features to have a mean of zero ensures that all features are centered around the same baseline, which helps prevent models from being biased toward features with larger numerical values. It also makes gradient-based optimization methods like gradient descent behave more efficiently by ensuring all features contribute equally to the cost function.

In [198]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### LOGIT

##### Packages

In [199]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

##### Training the Logistic Regression

In [200]:
log_reg = LogisticRegression()
log_reg.fit(X_train_scaled, y_train)

LogisticRegression()

##### Predictions

In [201]:
y_pred = log_reg.predict(X_test_scaled)

##### Evaluations of the accuracy of model

In [202]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.71


In [203]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.69      0.75      0.72       100
           1       0.73      0.67      0.70       100

    accuracy                           0.71       200
   macro avg       0.71      0.71      0.71       200
weighted avg       0.71      0.71      0.71       200



##### Estimating the PDs

In [204]:
df_R["PD_LR"] = log_reg.predict_proba(scaler.transform(X))[:, 1]
df_R


,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,y-categorical-default,educational level_bachelor degree,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR
0,name0,42,1,0,1224.0,1506.545590,-282.545590,0.230838,515.719196,1.345488,...,1,False,True,False,True,False,False,False,False,0.817767
1,name1,44,3,1,1896.0,1591.271515,304.728485,0.000000,1272.573333,1.008021,...,1,False,True,False,True,False,False,False,False,0.921881
2,name2,30,0,0,1033.0,803.987260,229.012740,0.000000,1153.501625,0.992579,...,0,False,True,False,True,False,False,False,False,0.433951
3,name3,41,0,1,6992.0,4667.171446,2324.828554,0.000000,4666.184651,1.193705,...,0,False,False,True,False,False,False,False,False,0.382267
4,name4,30,0,0,1070.0,868.404229,201.595771,0.000000,1410.257475,1.292106,...,0,False,True,False,True,False,False,False,False,0.545299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,56,0,0,4500.0,5459.236327,-959.236327,0.213164,1565.422592,0.989129,...,1,True,False,False,False,False,False,False,False,0.583401
996,name996,45,0,0,2436.0,1926.002775,509.997225,0.000000,2650.327579,0.684856,...,0,False,False,False,False,True,False,False,False,0.221781
997,name997,49,0,0,3500.0,2637.012330,862.987670,0.000000,3692.501195,0.717593,...,1,False,False,True,False,False,True,False,False,0.310580
998,name998,41,0,1,2550.0,2213.388587,336.611413,0.000000,2142.087015,1.371900,...,0,False,False,False,False,True,False,False,False,0.398481


### Using Neuronal Networks

##### Packages

In [205]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

##### Building the Neuronal Network

In [206]:
# We may try out:

# tanh: The hyperbolic tangent function outputs values between -1 and 1, making it useful for hidden layers 
# where you want activations that are zero-centered, which helps in faster convergence and avoids saturation for small inputs.
 
# relu: The Rectified Linear Unit activation function outputs zero for any negative input and passes positive values as they are.
# It is widely used in hidden layers for its simplicity and effectiveness, and helps avoid the vanishing gradient problem seen with functions like sigmoid and tanh.

# softmax: Softmax is typically used in the output layer for multi-class classification tasks. It converts the raw outputs into probabilities, 
# ensuring that the sum of all output values equals 1, representing the probability distribution over multiple classes.

# This is like having an input which is your variable X, then 32 neurons process the input features,
# using a function (in this case "tanh") to calculate the weights and transformations at each neuron. 
# The results are then passed to a subsequent layer with 16 neurons, where again "tanh" is applied to further transform the data.
# In the end, everything is passed through a final neuron that uses a "sigmoid" (logistic) function 
# to produce an output between [0, 1], representing a probability for binary classification.
model = Sequential([  # Each layer is run after the other, forming a linear stack of layers.
    # The first Dense layer applies 32 units (neurons) and uses the "tanh" activation function.
    # The input_shape corresponds to the number of features in the dataset (X_train_scaled).
    Dense(32, activation='tanh', input_shape=(X_train_scaled.shape[1],)),
    
    # The second Dense layer applies 16 units (neurons) and uses "tanh" activation function.
    # "tanh" ensures that the output of each neuron will be between -1 and 1, centering the activations.
    Dense(16, activation='tanh'),
    
    # The final Dense layer outputs a single value, which is the probability of the positive class.
    # Sigmoid activation squashes the output to a value between 0 and 1.
    # This is commonly used for binary classification, where the output is a probability of class 1.
    Dense(1, activation='sigmoid')
])

/Users/bonjour/opt/anaconda3/envs/bankgame/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


##### Compiling the model

In [207]:
# The model is being compiled with the following parameters:
# optimizer='adam': The Adam optimizer is being used. It is an adaptive learning rate optimization algorithm that 
#  combines the benefits of both AdaGrad and RMSProp, making it well-suited for most deep learning models.
# loss='binary_crossentropy': The loss function used is binary cross-entropy, which is appropriate for binary classification 
#  tasks where the output is a probability of belonging to one of two classes, which is the case of our y-variable
# metrics=['accuracy']: The model will track accuracy as the evaluation metric during training and testing, 
#  which measures the percentage of correct predictions.

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

##### Training the model

In [208]:
# The model is fit with X_train_scaled: This means the model is being trained on the scaled training data (X_train_scaled) 
# using the corresponding labels (y_train).

# Uses 25 epochs: An epoch refers to one full pass through the entire training dataset. 
# The model will train for 25 epochs, meaning it will go through the data 25 times to learn the optimal weights.

# It will use a batch size of 32: The model will train using 32 samples (or rows of data) at a time, and after processing 
# those 32, it updates the weights before moving on to the next 32 samples. 

# During training, the model's performance is periodically evaluated on the validation set (X_test_scaled and y_test) 
# to monitor overfitting and to adjust the training accordingly.

model_NN = model.fit(X_train_scaled, y_train, epochs=25, batch_size=32, validation_data=(X_test_scaled, y_test))

Epoch 1/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5278 - loss: 0.6945 - val_accuracy: 0.6200 - val_loss: 0.6654
Epoch 2/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6533 - loss: 0.6421 - val_accuracy: 0.6600 - val_loss: 0.6300
Epoch 3/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6839 - loss: 0.6139 - val_accuracy: 0.6700 - val_loss: 0.6181
Epoch 4/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6896 - loss: 0.5964 - val_accuracy: 0.6750 - val_loss: 0.6160
Epoch 5/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6792 - loss: 0.5966 - val_accuracy: 0.6950 - val_loss: 0.6123
Epoch 6/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6664 - loss: 0.6095 - val_accuracy: 0.7000 - val_loss: 0.6106
Epoch 7/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6703 - loss: 0.6030 - val_accuracy: 0.7050 - val_loss: 0.6091
Epoch 8/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6912 - loss: 0.5850 - val_accuracy: 0.7100 - val_loss:

##### We want to get the last accuracy of the last epoch and the minimal and highest accuracy values for comparison

In [209]:
train_accuracies_NN = model_NN.history['accuracy']

In [210]:
# final accuracy value
final_accuracy = train_accuracies_NN[-1]
# maximal accuracy value
min_accuracy = min(train_accuracies_NN)
# minimal accuracy value
max_accuracy = max(train_accuracies_NN)
# average accuracy value
average_accuracy = sum(train_accuracies_NN) / len(train_accuracies_NN)

In [211]:
# Print the results
print(f'Final accuracy: {final_accuracy:.4f}')
print(f'Minimum accuracy during training of the NN: {min_accuracy:.4f}')
print(f'Maximum accuracy during training of the NN: {max_accuracy:.4f}')
print(f'Average accuracy during training of the NN: {average_accuracy:.4f}')

Final accuracy: 0.7000
Minimum accuracy during training of the NN: 0.5525
Maximum accuracy during training of the NN: 0.7000
Average accuracy during training of the NN: 0.6785


##### Estimating the PDs

In [212]:
df_R["PD_NN"]= model.predict(scaler.transform(X))
df_R

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,educational level_bachelor degree,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR,PD_NN
0,name0,42,1,0,1224.0,1506.545590,-282.545590,0.230838,515.719196,1.345488,...,False,True,False,True,False,False,False,False,0.817767,0.862534
1,name1,44,3,1,1896.0,1591.271515,304.728485,0.000000,1272.573333,1.008021,...,False,True,False,True,False,False,False,False,0.921881,0.913223
2,name2,30,0,0,1033.0,803.987260,229.012740,0.000000,1153.501625,0.992579,...,False,True,False,True,False,False,False,False,0.433951,0.443681
3,name3,41,0,1,6992.0,4667.171446,2324.828554,0.000000,4666.184651,1.193705,...,False,False,True,False,False,False,False,False,0.382267,0.472169
4,name4,30,0,0,1070.0,868.404229,201.595771,0.000000,1410.257475,1.292106,...,False,True,False,True,False,False,False,False,0.545299,0.613007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,56,0,0,4500.0,5459.236327,-959.236327,0.213164,1565.422592,0.989129,...,True,False,False,False,False,False,False,False,0.583401,0.659916
996,name996,45,0,0,2436.0,1926.002775,509.997225,0.000000,2650.327579,0.684856,...,False,False,False,False,True,False,False,False,0.221781,0.215842
997,name997,49,0,0,3500.0,2637.012330,862.987670,0.000000,3692.501195,0.717593,...,False,False,True,False,False,True,False,False,0.310580,0.298309
998,name998,41,0,1,2550.0,2213.388587,336.611413,0.000000,2142.087015,1.371900,...,False,False,False,False,True,False,False,False,0.398481,0.411650


### Using Random Forests

##### Packages

In [213]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

##### Fitting the model

In [214]:
# Initialize the RandomForestClassifier with the following parameters:
# n_estimators=100: This sets the number of decision trees (estimators) in the forest. The model will train 100 individual trees and aggregate their results to make predictions.
# random_state=42: This ensures reproducibility by fixing the random seed used in the training process. 
# To get the same result even if the code is run multiple times (obviously this will only be affected by the random nature of our data)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

##### Accuracy of the model

In [215]:
# Predict class labels for X_test_scaled
y_pred = rf_model.predict(X_test_scaled)
# Calculate accuracy by comparing the predicted labels with our simulated data y
accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.4f}')

Accuracy: 0.6750


##### Estimating the PDs

In [216]:
# df_R = df_R.iloc[:len(X_test_scaled)]
# df_R["PD_RF"] = rf_model.predict_proba(X_test_scaled)[:, 1]
df_R["PD_RF"] = rf_model.predict_proba(scaler.transform(X))[:, 1] # needs to be checked
df_R

,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR,PD_NN,PD_RF
0,name0,42,1,0,1224.0,1506.545590,-282.545590,0.230838,515.719196,1.345488,...,True,False,True,False,False,False,False,0.817767,0.862534,0.91
1,name1,44,3,1,1896.0,1591.271515,304.728485,0.000000,1272.573333,1.008021,...,True,False,True,False,False,False,False,0.921881,0.913223,0.95
2,name2,30,0,0,1033.0,803.987260,229.012740,0.000000,1153.501625,0.992579,...,True,False,True,False,False,False,False,0.433951,0.443681,0.21
3,name3,41,0,1,6992.0,4667.171446,2324.828554,0.000000,4666.184651,1.193705,...,False,True,False,False,False,False,False,0.382267,0.472169,0.42
4,name4,30,0,0,1070.0,868.404229,201.595771,0.000000,1410.257475,1.292106,...,True,False,True,False,False,False,False,0.545299,0.613007,0.21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,56,0,0,4500.0,5459.236327,-959.236327,0.213164,1565.422592,0.989129,...,False,False,False,False,False,False,False,0.583401,0.659916,0.46
996,name996,45,0,0,2436.0,1926.002775,509.997225,0.000000,2650.327579,0.684856,...,False,False,False,True,False,False,False,0.221781,0.215842,0.21
997,name997,49,0,0,3500.0,2637.012330,862.987670,0.000000,3692.501195,0.717593,...,False,True,False,False,True,False,False,0.310580,0.298309,0.75
998,name998,41,0,1,2550.0,2213.388587,336.611413,0.000000,2142.087015,1.371900,...,False,False,False,True,False,False,False,0.398481,0.411650,0.32


### Using Logistic LASSO

#### Packages

In [217]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

#### Parameter of the regularization strenght

In [218]:
alpha = 0.01  # Regularization strength

#### Training the regression

In [219]:
# L1-regularized logistic regression (like lasso but for classification problems)
logistic_lasso = LogisticRegression(penalty='l1', solver='liblinear', C=1/alpha, random_state=42)

# Fit the model
logistic_lasso.fit(X_train_scaled, y_train)


LogisticRegression(C=100.0, penalty='l1', random_state=42, solver='liblinear')

In [220]:
logistic_lasso.coef_

array([[ 0.09892114,  0.63275237,  0.00146745, -0.05492213, -0.05314111,
        -0.04189966,  0.38100645, -0.33462339,  0.47800614, -0.08784368,
         0.23967049, -0.26472878,  0.33675302,  0.19893324,  0.14101745,
         0.14108481,  0.        , -0.11173204, -0.08960145,  0.05874774,
        -0.06360579]])

#### Making predictions with the regression

In [221]:
# Predict probabilities
y_pred_proba = logistic_lasso.predict_proba(X_test_scaled)[:, 1]  # Probabilities of class 1

# Apply threshold at 0.5 to get predicted class labels
y_pred_class = (y_pred_proba >= 0.5).astype(int)

Getting accuracy measures

In [222]:
# Accuracy by rule of PD >= 0.5 default and PD < 0.5 not default
y_pred_class = (y_pred_proba >= 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred_class)
print(f"Accuracy: {accuracy:.4f}")

# AUC score (for probability-based performance)
auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC: {auc:.4f}")

Accuracy: 0.7050
AUC: 0.7361


#### Now estiamting the PDs for our whole data set

In [223]:
df_R["PD_LL"] = logistic_lasso.predict_proba(scaler.transform(X))[:, 1]

### Now getting the general statistics of what I did, for all the four models: Logisitc regression, Neuronal networks, Random Forests and Logistic LASSO

In [224]:
df_R.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,debt-to-income ratio after credit,collateral,estimated seizable assets,y-categorical-default,PD_LR,PD_NN,PD_RF,PD_LL
count,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,44.78400,0.532000,0.535000,3612.168000,3329.364581,282.803419,0.106481,3164.658875,0.887691,43.308000,0.822880,27614.306000,35813.991376,0.498000,0.498238,0.499079,0.495730,0.498237
std,8.94804,0.737238,0.812056,2560.136893,2658.096965,1697.797528,0.227843,2698.608145,0.357755,15.973487,0.481638,22753.506356,27038.423536,0.500246,0.208070,0.212244,0.317873,0.209507
min,30.00000,0.000000,0.000000,500.000000,300.910488,-10698.896596,0.000000,184.585364,0.250541,12.000000,0.000000,0.000000,561.971358,0.000000,0.067767,0.095494,0.020000,0.065287
25%,37.00000,0.000000,0.000000,1749.750000,1475.213862,-246.916409,0.000000,1252.930880,0.570547,36.000000,0.466365,10715.500000,14873.622924,0.000000,0.336742,0.320445,0.180000,0.337087
50%,44.00000,0.000000,0.000000,2836.000000,2562.642602,241.563669,0.000000,2298.475855,0.898287,48.000000,0.798866,22630.500000,29007.856732,0.000000,0.470402,0.480632,0.470000,0.469862
75%,52.25000,1.000000,1.000000,4727.500000,4188.166526,851.489628,0.118208,4305.044396,1.187782,60.000000,1.149186,42660.500000,52468.411323,1.000000,0.644314,0.658619,0.810000,0.644104
max,60.00000,3.000000,4.000000,16008.000000,17432.801983,10725.032818,1.757010,18171.700020,1.499827,60.000000,2.905556,79960.000000,121104.313590,1.000000,0.985117,0.953043,1.000000,0.986229


## EAD (exposure at default)
It is the total amount at risk at the moment the borrower defaults

That is the amount fo the credit that has been unpaid at the moment of the calculation

In [225]:
df_R["EAD"] = df_R["credit: monthly amount"] * 12
df_R

,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD_LR,PD_NN,PD_RF,PD_LL,EAD
0,name0,42,1,0,1224.0,1506.545590,-282.545590,0.230838,515.719196,1.345488,...,True,False,False,False,False,0.817767,0.862534,0.91,0.819963,6188.630346
1,name1,44,3,1,1896.0,1591.271515,304.728485,0.000000,1272.573333,1.008021,...,True,False,False,False,False,0.921881,0.913223,0.95,0.924104,15270.879991
2,name2,30,0,0,1033.0,803.987260,229.012740,0.000000,1153.501625,0.992579,...,True,False,False,False,False,0.433951,0.443681,0.21,0.428811,13842.019505
3,name3,41,0,1,6992.0,4667.171446,2324.828554,0.000000,4666.184651,1.193705,...,False,False,False,False,False,0.382267,0.472169,0.42,0.381193,55994.215818
4,name4,30,0,0,1070.0,868.404229,201.595771,0.000000,1410.257475,1.292106,...,True,False,False,False,False,0.545299,0.613007,0.21,0.545554,16923.089699
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,56,0,0,4500.0,5459.236327,-959.236327,0.213164,1565.422592,0.989129,...,False,False,False,False,False,0.583401,0.659916,0.46,0.587598,18785.071098
996,name996,45,0,0,2436.0,1926.002775,509.997225,0.000000,2650.327579,0.684856,...,False,True,False,False,False,0.221781,0.215842,0.21,0.220391,31803.930948
997,name997,49,0,0,3500.0,2637.012330,862.987670,0.000000,3692.501195,0.717593,...,False,False,True,False,False,0.310580,0.298309,0.75,0.312219,44310.014340
998,name998,41,0,1,2550.0,2213.388587,336.611413,0.000000,2142.087015,1.371900,...,False,True,False,False,False,0.398481,0.411650,0.32,0.398474,25705.044181


## LGD (loss given default)

Example:

1. A borrower takes a loan of €10,000. 
2. They default after paying back €2,000, and the bank recovers €4,000 by seizing assets. 

That means:

Total Recovered = €2,000 (paid) + €4,000 (recovered from assets) = €6,000

Total Loss = €10,000 - €6,000 = €4,000

LGD = €4,000 (Total Loss)/ €10,000 (Loan Amount) = 40%



#### In our approximation 
LGD = (EAD - what can be sized)/EAD = 1 - (What can be seized/EAD)

In [226]:
def estimate_lgd(EAD, seizable_assets):
    lgd = 1 - (seizable_assets / EAD)
    return max(0, min(lgd, 1))  # The lgd should be between 0 and 1

In [227]:
df_R["LGD"] = df_R.apply(lambda row: estimate_lgd(row["EAD"], row["estimated seizable assets"]), axis=1) # This should be applied row by row

# We need here a Montecarlo to get the model with the highest accurracy results and they will be then added to the df, the others wont be added, but will remain part of the code.

## EL (expected loss)

EL=PD×LGD×EAD

In [228]:
df_R["EL"] = df_R["LGD"] * df_R["EAD"] * df_R["PD_NN"]

## We want here to estimate a suggested interest rate that should be charged to the customer

#### Inputs of the function

In [229]:
base_rate = 0.03  # central bank or risk-free rate
max_rate = 0.45 # maximum legal rate to be charged
PD = df_R["PD_NN"]  # Probability of default
LGD = df_R["LGD"]  # Loss given default
bank_margin = 0.02  # Bank's margin

#### Function

In [230]:
def calculate_suggested_rate(base_rate, max_rate, PD, LGD, margin):
    # Calculate the risk premium
    risk_premium = PD * LGD
    # Calculate the suggested rate
    suggested_rate = base_rate + risk_premium + margin
    return min(suggested_rate, max_rate)

#### Estimation of the calculated rate

In [231]:
df_R["Suggested interest rate"] = df_R.apply(
    lambda row: calculate_suggested_rate(base_rate, max_rate, row["PD_NN"], row["LGD"], bank_margin), axis=1
)

## Making one last description of the columns

In [232]:
df_R.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,...,estimated seizable assets,y-categorical-default,PD_LR,PD_NN,PD_RF,PD_LL,EAD,LGD,EL,Suggested interest rate
count,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,...,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,44.78400,0.532000,0.535000,3612.168000,3329.364581,282.803419,0.106481,3164.658875,0.887691,43.308000,...,35813.991376,0.498000,0.498238,0.499079,0.495730,0.498237,37975.906503,0.257296,5298.132317,0.160347
std,8.94804,0.737238,0.812056,2560.136893,2658.096965,1697.797528,0.227843,2698.608145,0.357755,15.973487,...,27038.423536,0.500246,0.208070,0.212244,0.317873,0.209507,32383.297739,0.319323,9585.768933,0.143236
min,30.00000,0.000000,0.000000,500.000000,300.910488,-10698.896596,0.000000,184.585364,0.250541,12.000000,...,561.971358,0.000000,0.067767,0.095494,0.020000,0.065287,2215.024363,0.000000,0.000000,0.050000
25%,37.00000,0.000000,0.000000,1749.750000,1475.213862,-246.916409,0.000000,1252.930880,0.570547,36.000000,...,14873.622924,0.000000,0.336742,0.320445,0.180000,0.337087,15035.170558,0.000000,0.000000,0.050000
50%,44.00000,0.000000,0.000000,2836.000000,2562.642602,241.563669,0.000000,2298.475855,0.898287,48.000000,...,29007.856732,0.000000,0.470402,0.480632,0.470000,0.469862,27581.710260,0.031199,440.991110,0.062365
75%,52.25000,1.000000,1.000000,4727.500000,4188.166526,851.489628,0.118208,4305.044396,1.187782,60.000000,...,52468.411323,1.000000,0.644314,0.658619,0.810000,0.644104,51660.532756,0.508690,6719.354604,0.254464
max,60.00000,3.000000,4.000000,16008.000000,17432.801983,10725.032818,1.757010,18171.700020,1.499827,60.000000,...,121104.313590,1.000000,0.985117,0.953043,1.000000,0.986229,218060.400244,0.928828,81970.355515,0.450000
